# Cycling Patterns in Flanders
**Modern Data Analytics - Group 2 - KU Leuven**

## Data Preparation

In [11]:
# Auto-reload modules so edits to src/ are picked up without restart
%load_ext autoreload
%autoreload 2

# Standard imports
import gc
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Make src/ importable from notebooks/
sys.path.insert(0, str(Path.cwd().parent))

from src.loaders import load_counts, load_sites, load_directions
from src.cleaning import clean_counts, merge_with_sites, flag_outliers
from src.features import add_calendar_features, add_covid_period, add_holidays
from src.transformers import CyclicalEncoder
from src.weather import fetch_open_meteo

# Global config
RANDOM_STATE = 42
DATA_DIR = Path.cwd().parent / "data" / "raw"
PROCESSED_DIR = Path.cwd().parent / "data" / "processed"
FIG_DIR = Path.cwd().parent / "figures"
FIG_DIR.mkdir(exist_ok=True)

# Plot styling
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)

print(f"Data dir: {DATA_DIR}")
print(f"Pandas: {pd.__version__} | sklearn: {__import__('sklearn').__version__}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Data dir: c:\Users\tobia\OneDrive\Bureaublad\Stat\sem2\Modern Data Analytics - Group 2\data\raw
Pandas: 2.2.3 | sklearn: 1.5.2


In [ ]:
# No kernel restart necessary after changes in src-folder
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.loaders import load_counts, load_sites, load_directions
from src.cleaning import clean_counts, merge_with_sites, flag_outliers
from src.features import add_calendar_features, add_covid_period, add_holidays
from src.transformers import CyclicalEncoder
from src.weather import fetch_open_meteo

DATA_DIR = Path.cwd().parent / "data" / "raw"
print(f"Data dir: {DATA_DIR}")

Data dir: c:\Users\tobia\OneDrive\Bureaublad\Stat\sem2\Modern Data Analytics - Group 2\data\raw


In [2]:
# Load counts
counts = load_counts(DATA_DIR)
counts.head()

Loaded 81 files | 42,685,516 rows


,site_id,richting,voertuig_type,van,tot,aantal
0,1,IN,FIETSERS,2019-08-01 00:00:00.0,2019-08-01 00:15:00.0,0.0
1,1,IN,FIETSERS,2019-08-01 00:15:00.0,2019-08-01 00:30:00.0,0.0
2,1,IN,FIETSERS,2019-08-01 00:30:00.0,2019-08-01 00:45:00.0,0.0
3,1,IN,FIETSERS,2019-08-01 00:45:00.0,2019-08-01 01:00:00.0,0.0
4,1,IN,FIETSERS,2019-08-01 01:00:00.0,2019-08-01 01:15:00.0,1.0


In [3]:
# Clean (filter + aggregate)
counts_clean = clean_counts(counts)
counts_clean.head()
print(f"After clean: {len(counts_clean):,} rows")

After clean: 10,419,659 rows


In [4]:
# Load sites
sites = load_sites(DATA_DIR)
sites.head()
sites.columns.tolist()

['site_id',
 'naam',
 'lon',
 'lat',
 'gemeente',
 'beheerder',
 'paalnummer',
 'code',
 'locatie',
 'interval_min',
 'installatie_datum']

In [5]:
# Merge
df = merge_with_sites(counts_clean, sites)
df.head()

,site_id,richting,timestamp,count,naam,lon,lat,gemeente,beheerder,paalnummer,code,locatie,interval_min,installatie_datum
0,1,IN,2019-08-01 00:00:00,0,100046096,4.456122,50.916183,Machelen,Vlaamse Overheid A. Wegen enVerkeer,T2110002,AWV212,Machelen,15,2019-08-22
1,1,IN,2019-08-01 01:00:00,1,100046096,4.456122,50.916183,Machelen,Vlaamse Overheid A. Wegen enVerkeer,T2110002,AWV212,Machelen,15,2019-08-22
2,1,IN,2019-08-01 02:00:00,1,100046096,4.456122,50.916183,Machelen,Vlaamse Overheid A. Wegen enVerkeer,T2110002,AWV212,Machelen,15,2019-08-22
3,1,IN,2019-08-01 03:00:00,0,100046096,4.456122,50.916183,Machelen,Vlaamse Overheid A. Wegen enVerkeer,T2110002,AWV212,Machelen,15,2019-08-22
4,1,IN,2019-08-01 04:00:00,0,100046096,4.456122,50.916183,Machelen,Vlaamse Overheid A. Wegen enVerkeer,T2110002,AWV212,Machelen,15,2019-08-22


In [ ]:
# Free memory
del counts, counts_clean
import gc; gc.collect()

In [6]:
# Outliers & features
df = flag_outliers(df)
df = add_calendar_features(df)
df = add_covid_period(df)
df = add_holidays(df)
df.head()

Flagged 101,965 outliers (0.98%)


,site_id,richting,timestamp,count,naam,lon,lat,gemeente,beheerder,paalnummer,...,is_outlier,hour,day_of_week,month,year,is_weekend,is_morning_rush,is_evening_rush,covid_period,is_holiday
0,1,IN,2019-08-01 00:00:00,0,100046096,4.456122,50.916183,Machelen,Vlaamse Overheid A. Wegen enVerkeer,T2110002,...,False,0,3,8,2019,False,False,False,pre_covid,False
1,1,IN,2019-08-01 01:00:00,1,100046096,4.456122,50.916183,Machelen,Vlaamse Overheid A. Wegen enVerkeer,T2110002,...,False,1,3,8,2019,False,False,False,pre_covid,False
2,1,IN,2019-08-01 02:00:00,1,100046096,4.456122,50.916183,Machelen,Vlaamse Overheid A. Wegen enVerkeer,T2110002,...,False,2,3,8,2019,False,False,False,pre_covid,False
3,1,IN,2019-08-01 03:00:00,0,100046096,4.456122,50.916183,Machelen,Vlaamse Overheid A. Wegen enVerkeer,T2110002,...,False,3,3,8,2019,False,False,False,pre_covid,False
4,1,IN,2019-08-01 04:00:00,0,100046096,4.456122,50.916183,Machelen,Vlaamse Overheid A. Wegen enVerkeer,T2110002,...,False,4,3,8,2019,False,False,False,pre_covid,False


In [12]:
# Make parquet file
processed_path = Path.cwd().parent / "data" / "processed" / "cycling_features.parquet"
df.to_parquet(processed_path, index=False)
print(f"Saved {len(df):,} rows to {processed_path}")
df = pd.read_parquet(processed_path)

Saved 10,419,659 rows to c:\Users\tobia\OneDrive\Bureaublad\Stat\sem2\Modern Data Analytics - Group 2\data\processed\cycling_features.parquet


## Exploratory Data Analysis